# Day06 — inference pipeline и сохранение модели

## D6-01 — frozen inputs и notebook setup

**Цель:** подготовить проверенный read-only reference для будущего inference bundle. Входы: Day02 processed/provenance, Day03 config/manifest, Day04 selection/config/validation reference и Day05 frozen protocol/final cache. Выход: типизированный `InferenceReference` с train/validation таблицами и frozen contract.

**Теория.** Config описывает алгоритм и параметры, а fitted estimator хранит выученные vocabulary/IDF и веса classifier. Сохранённые predictions — ответы на конкретные строки; они не позволяют классифицировать новый текст. Поэтому Day06 позже явно восстановит модель на прежнем train, проверит validation replay и сохранит fitted Pipeline. Здесь модели ещё нет.

Поток будущего inference: **raw texts `(N,)` → `preprocess_text` (`minimal-v1`) → clean texts `(N,)` → fitted TF-IDF `(N, V)` sparse → classifier → predictions `(N,)`**. Vocabulary/IDF fit только на train. Ни validation, ни test не добавляются к fit. Test не используется для новых model calls.

Сейчас loaders проверяют hashes, схемы, полные параметры, выбор по CV, версии, IDs/labels/order и непересечение groups. Read-only final-cache reader сверяет сохранённые Day05 outputs; нового fit, predict, CV, download или запуска final evaluation нет. Reference отдаёт только train/validation, без test texts/predictions.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src/finnews_sentiment").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src/finnews_sentiment").is_dir():
    raise RuntimeError("Run from the project root or notebooks directory")
sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.models.inference_artifact import (
    InferenceReference, load_inference_reference,
)

reference: InferenceReference = load_inference_reference(ROOT)
contract = reference.contract
print("Candidate:", contract.candidate_id)
print("Fit policy:", contract.fit_policy)
print("Preprocessing:", contract.preprocessing_policy)
print("Class order:", contract.class_order)
print("Label mapping:", contract.label_mapping)
print("Seed:", contract.random_state)
print("Schema for future bundle:", contract.schema_version)
print("Full effective parameters:", contract.pipeline_parameters)

Candidate: linear_svc
Fit policy: day03_train_only
Preprocessing: minimal-v1
Class order: ('negative', 'neutral', 'positive')
Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Seed: 42
Schema for future bundle: day06-inference-v1
Full effective parameters: {'tfidf': {'analyzer': 'word', 'binary': False, 'decode_error': 'strict', 'dtype': 'float64', 'encoding': 'utf-8', 'input': 'content', 'lowercase': False, 'max_df': 1.0, 'max_features': 5000, 'min_df': 1, 'ngram_range': [1, 2], 'norm': 'l2', 'preprocessor': None, 'smooth_idf': True, 'stop_words': None, 'strip_accents': None, 'sublinear_tf': False, 'token_pattern': '(?u)\\b\\w\\w+\\b', 'tokenizer': None, 'use_idf': True, 'vocabulary': None}, 'classifier': {'C': 1.0, 'class_weight': None, 'dual': 'auto', 'fit_intercept': True, 'intercept_scaling': 1, 'loss': 'squared_hinge', 'max_iter': 5000, 'multi_class': 'ovr', 'penalty': 'l2', 'random_state': 42, 'tol': 0.0001, 'verbose': 0}}


In [2]:
print("Train DataFrame:", reference.train.shape)
print("Validation DataFrame:", reference.validation.shape)
print("Validation predictions reference:", reference.validation_predictions.shape)
print("Day05 recorded feature count:", reference.day05_n_features)
print("Day05 cached replay:", reference.day05_validation_replay)
display(reference.split_summary)
display(reference.train.head(3))
display(reference.validation_predictions.head(3))

Train DataFrame: (2210, 5)
Validation DataFrame: (553, 5)
Validation predictions reference: (553, 3)
Day05 recorded feature count: 5000
Day05 cached replay: exact_match


,rows,groups,negative,neutral,positive
split,,,,,
train,2210,2206,269,1374,567
validation,553,552,67,344,142
test,690,690,84,428,178


,source_row_id,text,sentiment,text_clean,label_id
1,1,With the new production plant the company woul...,positive,with the new production plant the company woul...,2
4,4,Operating profit rose to EUR 13.1 mn from EUR ...,positive,operating profit rose to eur 13.1 mn from eur ...,2
5,5,"Operating profit totalled EUR 21.1 mn , up fro...",positive,"operating profit totalled eur 21.1 mn , up fro...",2


,source_row_id,true_label_id,pred_label_id
0,2,2,2
1,3,2,2
2,11,2,2


### Источники и проверенные contracts

Таблицы train/validation имеют пять columns: `source_row_id`, `text`, `sentiment`, `text_clean`, `label_id`. Prediction reference содержит три columns: `source_row_id`, `true_label_id`, `pred_label_id`. Их порядок совпадает с validation manifest. Split summary показывает только counts по splits, без test texts.

Каждая строка сохраняет вес в обучении; same-label duplicates остаются в одном split. Непересечение exact-text groups не доказывает независимость похожих предложений. SHA-256 защищает согласованность bytes, semantic checks — смысл схем/IDs/параметров. Implementation hash preprocessing фиксируется **сейчас** для будущего bundle: он не доказывает, какие bytes исходника использовались в утраченной Day05 модели.

In [3]:
sources = pd.DataFrame(
    [{"path": path, "sha256": checksum}
     for path, checksum in sorted(contract.input_checksums.items())]
)
print("Verified input/final files:", len(sources))
display(sources)
print("Preprocessing implementation SHA-256:", contract.preprocessing_sha256)
print("Frozen runtime versions:", contract.runtime_versions)

assert reference.train["source_row_id"].tolist() == list(contract.train_source_row_ids)
assert reference.validation["source_row_id"].tolist() == list(contract.validation_source_row_ids)
assert reference.validation_predictions["source_row_id"].tolist() == list(contract.validation_source_row_ids)
assert reference.validation_predictions["true_label_id"].tolist() == reference.validation["label_id"].tolist()
assert set(contract.train_source_row_ids).isdisjoint(contract.validation_source_row_ids)
print("IDs/order/truth aligned; no fitted Day06 model or new predictions created.")

Verified input/final files: 28


,path,sha256
0,configs/dataset.json,fc43038477b9beeb491c505244b7b0a403b8da46f062f6...
1,configs/experiments_day04.json,d9830dd3805252cc523a2fcb6fc22a28638e2c59d31fee...
2,configs/train_baseline.json,8ffeb9230ae0cbffd6c00939694d028370ea5e825eaabf...
3,data/processed/day03/splits.csv,b51d363cac4c00f18d760aa72d3ca6f5272becced6be45...
4,data/processed/financial_phrasebank_75agree.csv,39dda898a281dc3294be081917b0681524e73edb60f009...
5,reports/day02/provenance.json,8d9e039f9ff7a76efb727a3e21ce2f07232aedb7b38600...
6,reports/day03/config.json,90b5abb6d13b209e69b989d052e9da87bef50a9e936cae...
7,reports/day03/metrics.json,727282dc6860f800b0ed05bcb1d17bef7fff007db424c2...
8,reports/day03/provenance.json,73f72d587bff42974d4eed6274d6886f03843d264aaf7e...
9,reports/day03/split_summary.json,8fe5b294800246fa9883def1a9fc615831385ed2c689a2...


Preprocessing implementation SHA-256: a08d569712d86603ee966aa338d931d3dfe805d66c3a9a5f308704414487f4fd
Frozen runtime versions: {'python': '3.14.6', 'pandas': '3.0.6', 'numpy': '2.5.3', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'matplotlib': '3.11.2', 'joblib': '1.6.0', 'threadpoolctl': '3.7.0'}
IDs/order/truth aligned; no fitted Day06 model or new predictions created.


### Интерпретация и самопроверка

На реальных локальных artifacts: train **2210 строк / 2206 groups**, validation **553 строки / 552 groups**. Day05 записал `V=5000` и `exact_match` validation replay. Это историческая metadata проверенного cache; **в этом section новый Day06 replay ещё не выполнен**, fitted bundle не создан. Раздел D6-02 ниже отдельно выполняет train-only build и fresh validation replay; сохранение модели относится к D6-03.

`FrozenInferenceContract` описывает проверенную конфигурацию и provenance. В следующем разделе D6-02 `BundleMetadata` и `InferenceBundle` добавляют успешный fresh replay и fitted Pipeline. Сигнатуры build/save/load зафиксированы в docstring `inference_artifact.py`; исполняемых заглушек нет. Неизвестный или повреждённый reference вызывает ошибку с путём, а не скрытую подготовку данных или переобучение.

1. Почему saved predictions и config не заменяют fitted estimator?
2. Почему test texts не передаются будущему builder, хотя проверяются hashes final cache?
3. Чем cached Day05 replay отличается от будущего fresh Day06 replay?
4. Какую утечку создаст fit TF-IDF на train + validation?
5. Почему одинаковая длина таблиц ещё не доказывает alignment по source IDs?

## D6-02 — train-only build и точный validation replay

**Цель:** восстановить выбранный по train-only CV алгоритм на прежних train строках. Вход — проверенный `InferenceReference`; выход — `InferenceBundle` с fitted Pipeline **в памяти**, fresh replay evidence и размерами обучения. Запись joblib/manifest относится к D6-03.

**Теория.** Сначала заново проверяются frozen artifacts и неизменность переданного reference, затем `preprocess_text(raw) == text_clean` на всех train/validation строках. Прежний builder создаёт candidate из selection; его полные параметры, включая defaults, сверяются с frozen protocol. TF-IDF vocabulary/IDF и classifier обучаются только на clean train в исходном manifest order, с сохранением duplicates.

После fit один вызов `predict` на validation должен точно воспроизвести все Day04 pred IDs; source IDs, true labels и порядок проверены до fit. `ConvergenceWarning`, несовпадение параметров, preprocessing или replay прекращают build. Validation не добавляется к train; test texts не передаются model calls. Vocabulary/IDF/classes не должны изменяться при replay.

**Этот checkpoint явно запускает train-only обучение при каждом исполнении.** После D6-03 build станет отдельной командой, а этот section будет читать metadata готового bundle без fit.

In [4]:
from finnews_sentiment.models.inference_artifact import (
    InferenceBundle, build_inference_bundle,
)

built_bundle: InferenceBundle = build_inference_bundle(reference)
build_metadata = built_bundle.metadata
print("Candidate:", build_metadata.contract.candidate_id)
print("Fit policy:", build_metadata.contract.fit_policy)
print("Train rows:", build_metadata.n_train)
print("Validation rows:", build_metadata.n_validation)
print("TF-IDF train dimensions:", (build_metadata.n_train, build_metadata.n_features))
print("Fresh Day06 validation replay:", build_metadata.validation_replay)
print("Day04 validation reference SHA-256:", build_metadata.validation_reference_sha256)
print("Bundle exists in memory only; no model/report files written.")

Candidate: linear_svc
Fit policy: day03_train_only
Train rows: 2210
Validation rows: 553
TF-IDF train dimensions: (2210, 5000)
Fresh Day06 validation replay: exact_match
Day04 validation reference SHA-256: 71d74d84970a956d80306899b0928431177ffcfa7498f3f4fe6e275245a2f987
Bundle exists in memory only; no model/report files written.


In [5]:
assert build_metadata.n_train == len(reference.train)
assert build_metadata.n_validation == len(reference.validation_predictions)
assert build_metadata.n_features == reference.day05_n_features
assert build_metadata.contract == reference.contract
assert build_metadata.validation_replay == "exact_match"
assert len(built_bundle.pipeline.named_steps["tfidf"].get_feature_names_out()) == build_metadata.n_features
print("Fitted class IDs:", built_bundle.pipeline.classes_)
print("Train duplicate rows retained:", int(reference.train["text_clean"].duplicated().sum()))
display(reference.validation_predictions.head(3))
print("All validation predictions matched this verified reference, not just the displayed rows.")

Fitted class IDs: [0 1 2]
Train duplicate rows retained: 4


,source_row_id,true_label_id,pred_label_id
0,2,2,2
1,3,2,2
2,11,2,2


All validation predictions matched this verified reference, not just the displayed rows.


### Интерпретация и самопроверка D6-02

На реальных данных candidate `linear_svc` обучается на **2210 train строках / 2206 groups**; четыре duplicate rows сверх первого сохранены. TF-IDF имеет **5000 features**, sparse train dimensions `(2210, 5000)`, validation predictions — `(553,)`. Все **553** Day04 predictions воспроизведены с точным alignment; `ConvergenceWarning` отсутствует.

Fresh replay подтверждает воспроизведение замороженного алгоритма, параметров и fit policy на этом validation. Он не доказывает идентичность весов утраченного fitted объекта Day05 и не является новой независимой оценкой качества: validation уже использовалась ранее. На test нет нового predict/transform/score; selection и параметры не меняются.

В отличие от cached replay metadata в D6-01, `build_metadata.validation_replay` относится к только что обученному Day06 Pipeline. Возвращённый объект можно передать будущему save API; сериализация и load ещё не реализованы.

1. Что доказывает exact validation replay и почему он не доказывает побитовую идентичность Day05 весов?
2. Почему validation-only vocabulary token не должен появиться в `vocabulary_`?
3. Почему нельзя удалить четыре retained train duplicates при восстановлении модели?
4. Чем этот явный build отличается от будущего inference, который не вызывает fit?
5. Почему `ConvergenceWarning` останавливает восстановление, хотя sklearn уже вернул estimator?